## Corrective RAG with LangChain

**Corrective RAG** adds a quality-control loop to standard RAG. Instead of blindly passing all retrieved documents to the LLM, each document is **graded for relevance** first. If too few relevant documents are found, the query is **rewritten** and retrieval is retried.

```
Query → Retrieve docs
             ↓
      Grade each doc (LLM: relevant / irrelevant)
             ↓
   ┌─────────┴──────────┐
   │ enough relevant?   │
   │   YES              │   NO
   ↓                    ↓
Answer from             Rewrite query (LLM)
relevant docs only      → Re-retrieve
                        → Grade again
                        → Answer (best effort)
```

### 1.1 Setup (LLM, Embeddings, Pinecone)

In [1]:
# STEP 1 — Setup

import os
import time
from datetime import datetime
from dotenv import load_dotenv

from pinecone import Pinecone, ServerlessSpec
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

# Pinecone index
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
INDEX_NAME = "coffee-hybrid"

if INDEX_NAME in pc.list_indexes().names():
    print(f"Deleting existing index: {INDEX_NAME}")
    pc.delete_index(INDEX_NAME)
    for _ in range(30):
        if INDEX_NAME not in pc.list_indexes().names():
            print("✅ Deleted.")
            break
        time.sleep(1)

pc.create_index(
    name=INDEX_NAME,
    dimension=384,
    metric="dotproduct",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

print("⏳ Waiting for index to be ready...")
for _ in range(60):
    if pc.describe_index(INDEX_NAME).status["ready"]:
        print("✅ Index ready.")
        break
    time.sleep(1)

index = pc.Index(INDEX_NAME)

c:\Users\Lucifer\anaconda3\envs\rag101_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Deleting existing index: coffee-hybrid
✅ Deleted.
⏳ Waiting for index to be ready...
✅ Index ready.


In [2]:
# LLM + Embeddings

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0.2,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
)

C:\Users\Lucifer\AppData\Local\Temp\ipykernel_117324\340692477.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [3]:
# Vector Stores + Memory

from langchain_core.documents import Document

vs_docs = PineconeVectorStore(
    index_name=INDEX_NAME, embedding=embeddings, text_key="text"
)

USER_ID = "user_001"
MEM_NS = f"mem_{USER_ID}"
vs_mem = PineconeVectorStore(
    index_name=INDEX_NAME, embedding=embeddings, text_key="text", namespace=MEM_NS
)

def add_memory(text: str, **meta):
    meta.setdefault("kind", "memory")
    meta.setdefault("ts", int(datetime.utcnow().timestamp()))
    vs_mem.add_documents([Document(page_content=text, metadata=meta)])
    print(f'  [Memory saved] "{text[:80]}"')

In [4]:
# STEP 2 — Document Ingestion (Load, Clean, Store)

from pathlib import Path
from langchain_community.document_loaders import UnstructuredHTMLLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import re

def remove_disclaimer(text):
    """Remove common disclaimer text that appears on all pages."""
    pattern = r"Disclaimer:[\s\S]*?before trying new herbs or routines\.\s*"
    return re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

html_dir = Path("coffee_pages")
html_docs = []

for fp in html_dir.glob("*.html"):
    try:
        loaded = UnstructuredHTMLLoader(str(fp)).load()
        for d in loaded:
            meta = dict(d.metadata or {})
            meta.pop("text", None)
            meta["source"] = str(fp)
            content = remove_disclaimer(d.page_content)
            if content:
                html_docs.append(Document(page_content=content, metadata=meta))
    except Exception as e:
        print(f"[WARN] Skipping {fp.name}: {e}")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
chunks = splitter.split_documents(html_docs)

vs_docs.add_documents(chunks)
print(f"✅ Stored {len(chunks)} chunks in '{INDEX_NAME}' (disclaimer removed)")

✅ Stored 53 chunks in 'coffee-hybrid' (disclaimer removed)


### 1.2 Retrievers

In [5]:
# STEP 3 — Retrievers
# Kept simple (dense + MMR) — the corrective grading is the focus of this notebook.

ret_docs = vs_docs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10, "fetch_k": 40, "lambda_mult": 0.7}
)

ret_mem = vs_mem.as_retriever(search_kwargs={"k": 6})

### 1.3 Document Grader — The Corrective Step

This is what makes **Corrective RAG** different from standard RAG:

| | Standard RAG | Corrective RAG |
|---|---|---|
| **Retrieved docs** | All passed to LLM | Each doc graded for relevance |
| **Irrelevant docs** | Pollute context → noisy answers | Filtered out before answering |
| **When nothing matches** | LLM hallucinates or gives vague answer | Query is rewritten and retried |
| **Quality control** | None | LLM-as-a-judge per document |

The grader uses the LLM to evaluate each document with a simple yes/no relevance check. This costs extra LLM calls but dramatically improves answer quality by ensuring only on-topic documents reach the final prompt.

In [6]:
# STEP 4 — Document Grader

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

grader_parser = JsonOutputParser()
grader_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a relevance grader. Given a QUESTION and a DOCUMENT, decide if the "
     "document contains information relevant to answering the question.\n"
     "Return ONLY JSON with keys:\n"
     "  relevant: true or false\n"
     "  reason: one short sentence explaining why\n"
     "{format}"),
    ("human", "QUESTION: {question}\n\nDOCUMENT:\n{document}")
])
grader_chain = grader_prompt | llm | grader_parser


def grade_documents(question, docs, verbose=False):
    """Grade each document for relevance. Returns (relevant, irrelevant) lists."""
    relevant, irrelevant = [], []
    for d in docs:
        try:
            result = grader_chain.invoke({
                "question": question,
                "document": d.page_content[:500],
                "format": grader_parser.get_format_instructions()
            })
            is_relevant = result.get("relevant", False)
        except Exception:
            is_relevant = True  # on parse error, keep the doc
            result = {"reason": "(grader parse error, kept by default)"}

        if is_relevant:
            relevant.append(d)
        else:
            irrelevant.append(d)

        if verbose:
            status = "✅" if is_relevant else "❌"
            src = d.metadata.get("source", "?")[-40:]
            print(f"    {status} {src} — {result.get('reason', '')}")

    return relevant, irrelevant

### 1.4 Query Rewriter

When the grader finds most retrieved documents are irrelevant, the original query may be too vague or use terms the corpus doesn't contain. The **query rewriter** asks the LLM to rephrase the question to be more specific and search-friendly before retrying retrieval.

In [7]:
# STEP 5 — Query Rewriter

from langchain_core.output_parsers import StrOutputParser

rewriter_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query rewriter. The user's question did not retrieve good results "
     "from a coffee-related knowledge base. Rewrite the question to be more specific, "
     "use alternative keywords, and make it more search-friendly.\n"
     "Return ONLY the rewritten question, nothing else."),
    ("human", "Original question: {question}")
])
rewriter_chain = rewriter_prompt | llm | StrOutputParser()

### 1.5 Answer Prompt + Helpers

In [8]:
# STEP 6 — Answer Prompt + Helpers

from langchain_core.prompts import MessagesPlaceholder
from langchain.schema import HumanMessage, AIMessage

def format_block(docs):
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source") or d.metadata.get("kind") or f"S{i}"
        snip = (d.page_content or "").strip().replace("\n", " ")
        lines.append(f"[S{i}] {src}\n{snip}")
    return "\n\n".join(lines)

def format_memory(docs):
    """Format memory docs as [M#] snippets."""
    lines = []
    for i, d in enumerate(docs, 1):
        snip = (d.page_content or "").strip().replace("\n", " ")
        lines.append(f"[M{i}] {snip}")
    return "\n".join(lines) if lines else "(no saved memories)"

def approx_tokens(s): return max(1, len(s or "") // 4)

def cap_docs(docs, max_tokens=1200):
    kept, total = [], 0
    for d in docs:
        n = approx_tokens(d.page_content)
        if total + n > max_tokens: break
        kept.append(d); total += n
    return kept

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are Askly, a helpful assistant.\n"
     "Answer using CONTEXT (corpus documents) and USER MEMORY (personal preferences).\n"
     "If info is missing, say so honestly. Cite corpus as [S1],[S2] and memory as [M1],[M2].\n"
     "When the user asks about their preferences, ALWAYS check USER MEMORY first.\n"
     "Be concise and actionable."),
    MessagesPlaceholder("history"),
    ("system", "USER MEMORY:\n{memory}"),
    ("system", "CONTEXT:\n{context}"),
    ("human", "{question}")
])
answer_chain = answer_prompt | llm | StrOutputParser()

### 1.6 Corrective RAG Pipeline & Demo

The pipeline below implements the full corrective loop:

1. **Retrieve** — fetch candidate docs from Pinecone
2. **Grade** — LLM evaluates each doc for relevance (yes/no + reason)
3. **Decision** — if `≥ min_relevant` docs pass → answer; otherwise → rewrite
4. **Rewrite** — LLM rephrases the query to be more search-friendly
5. **Re-retrieve + Re-grade** — second attempt with the rewritten query
6. **Answer** — final answer from whatever relevant docs were found (or honest "I don't know")

Memory is **always fetched separately** and passed into the `{memory}` placeholder.

In [11]:
# STEP 7 — Corrective RAG Pipeline + Scripted Demo

def corrective_answer(question, history=None, verbose=True, min_relevant=2):
    """
    Corrective RAG: retrieve → grade → (rewrite if needed) → answer.
    Memory is always fetched separately.
    """
    history = history or []

    # 1) Retrieve corpus docs
    pool = ret_docs.invoke(question)
    if verbose:
        print(f"  📥 RETRIEVE: {len(pool)} docs fetched")

    # 2) Grade each doc
    if verbose:
        print(f"  📝 GRADING (attempt 1):")
    relevant, irrelevant = grade_documents(question, pool, verbose=verbose)
    if verbose:
        print(f"  📊 RESULT: {len(relevant)}/{len(pool)} relevant, {len(irrelevant)}/{len(pool)} irrelevant")

    # 3) If not enough relevant docs, rewrite and retry
    rewritten_q = None
    if len(relevant) < min_relevant:
        rewritten_q = rewriter_chain.invoke({"question": question})
        if verbose:
            print(f"\n  🔄 REWRITE: \"{question}\" \u2192 \"{rewritten_q}\"")

        # Re-retrieve with rewritten query
        pool2 = ret_docs.invoke(rewritten_q)
        if verbose:
            print(f"  📥 RE-RETRIEVE: {len(pool2)} docs fetched")
            print(f"  📝 GRADING (attempt 2):")

        relevant2, irrelevant2 = grade_documents(rewritten_q, pool2, verbose=verbose)
        if verbose:
            print(f"  📊 RESULT: {len(relevant2)}/{len(pool2)} relevant, {len(irrelevant2)}/{len(pool2)} irrelevant")

        # Merge: use rewritten results if better, otherwise combine
        if len(relevant2) > len(relevant):
            relevant = relevant2
        else:
            # Deduplicate by page_content
            seen = {d.page_content for d in relevant}
            for d in relevant2:
                if d.page_content not in seen:
                    relevant.append(d)
                    seen.add(d.page_content)

    # 4) Always fetch memory separately
    mem_docs = ret_mem.invoke(question)
    if verbose:
        if mem_docs:
            for i, d in enumerate(mem_docs, 1):
                print(f"  🧠 MEMORY[{i}]: {d.page_content[:80]}")
        else:
            print(f"  🧠 MEMORY: (no saved memories)")

    # 5) Cap and answer
    final_docs = cap_docs(relevant, max_tokens=1200)
    if verbose:
        total_tokens = sum(approx_tokens(d.page_content) for d in final_docs)
        print(f"  📈 FINAL: {len(final_docs)} relevant docs passed to LLM ({total_tokens} tokens)")

    ctx = format_block(final_docs)
    mem = format_memory(mem_docs)

    answer = answer_chain.invoke({
        "history": history,
        "context": ctx,
        "memory": mem,
        "question": question,
    })

    return answer, final_docs, mem_docs, rewritten_q


# ── DEMO: 6-turn corrective RAG conversation ─────────────────────────────

DEMO = [
    # Turn 1 — direct match: grader should pass most docs
    ("ask", "What are the ingredients in ashwagandha coffee?"),

    # Turn 2 — off-topic for corpus: should trigger rewrite + retry
    ("ask", "What is the history of Ethiopian coffee ceremonies?"),

    # Save a preference to memory
    ("mem", "I like drinks with cardamom"),

    # Turn 3 — memory + corpus: grader filters non-spice docs
    ("ask", "Suggest a spiced coffee for me"),

    # Turn 4 — medical question: corpus lacks this, rewrite still fails
    ("ask", "How do adaptogens interact with blood pressure medication?"),

    # Turn 5 — history + memory synthesis
    ("ask", "What coffees have we discussed that might suit my taste?"),
]

history = []
print("\n\ud83e\uddea CORRECTIVE RAG \u2014 Scripted Demo\n" + "=" * 70)

for kind, text in DEMO:

    if kind == "mem":
        print(f"\n\ud83d\udccc SAVING MEMORY: \"{text}\"")
        add_memory(text)
        continue

    turn = len(history) // 2 + 1
    # print(f"\n{'\u2500' * 70}")
    print(f"  Turn {turn}  |  History: {len(history)} messages")
    print(f"  Q: {text}")
    # print(f"{'\u2500' * 70}")

    history.append(HumanMessage(content=text))

    answer, ctx_docs, mem_docs, rewritten = corrective_answer(
        text, history=history, verbose=True
    )

    if rewritten:
        print(f"  \u2139\ufe0f  Query was rewritten for better retrieval")

    print(f"\n  A: {answer}")

    history.append(AIMessage(content=answer))

    print(f"\n  \ud83d\udcdd HISTORY SNAPSHOT ({len(history)} messages):")
    for msg in history:
        role = "You  " if isinstance(msg, HumanMessage) else "Askly"
        snippet = msg.content[:70].replace("\n", " ")
        ellipsis = "..." if len(msg.content) > 70 else ""
        print(f"      {role}: {snippet}{ellipsis}")

print(f"\n{'=' * 70}")
print("\u2705 Corrective RAG demo complete.")

  📥 RETRIEVE: 10 docs fetched
  📝 GRADING (attempt 1):
    ✅ shwagandha_coffee_adaptogenic_latte.html — The document lists milk and coffee as ingredients and mentions ashwagandha as a key component of ashwagandha coffee.
    ✅ _ashwagandha_coffee_adaptogen_blend.html — The document lists the ingredients for 'Mushroom & Ashwagandha Coffee', which includes ashwagandha powder.
    ❌ shwagandha_coffee_adaptogenic_latte.html — The document discusses the cultural context and usage tips for ashwagandha powder in coffee but does not list any specific ingredients for ashwagandha coffee.
    ✅ shwagandha_coffee_adaptogenic_latte.html — The document provides a list of ingredients for making ashwagandha coffee, including ashwagandha powder, coffee, and other components.
    ❌ _masala_coffee_spiced_indian_coffee.html — The document describes the ingredients for Masala Coffee, not ashwagandha coffee.
    ✅ shwagandha_coffee_adaptogenic_latte.html — The document describes how to make ashwagandha coff

Exception in callback BaseAsyncIOLoop._handle_events(1376, 1)
handle: <Handle BaseAsyncIOLoop._handle_events(1376, 1)>
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 1652-1653: surrogates not allowed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Lucifer\anaconda3\envs\rag101_311\Lib\site-packages\jupyter_client\session.py", line 143, in orjson_packer
    return orjson.dumps(obj, default=json_default, option=option)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: str is not valid UTF-8: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\Lucifer\anaconda3\envs\rag101_311\Lib\site-packages\jupyter_client\session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec ca